<a href="https://colab.research.google.com/github/yuweima0506-pixel/IEEE-CIS-Fraud-Detection/blob/main/baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

ieee_fraud_detection_path = kagglehub.competition_download('ieee-fraud-detection')

print('Data source import complete.')


In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
import xgboost as xgb
import shap
import warnings

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/ieee-fraud-detection/sample_submission.csv
/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv
/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv
/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv


In [ ]:
train_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_transaction.csv')
train_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/train_identity.csv')
test_transaction = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_transaction.csv')
test_identity = pd.read_csv('/kaggle/input/competitions/ieee-fraud-detection/test_identity.csv')
train = train_transaction.merge(train_identity, on='TransactionID', how = 'left')
test = test_transaction.merge(test_identity, on='TransactionID', how = 'left')

In [ ]:

# Categorical Features: ProductCD card1 - card6 addr1, addr2 P_emaildomain R_emaildomain M1 - M9

#-- identity
#Categorical Features: DeviceType DeviceInfo id_12 - id_38
cat_fea = ['ProductCD','card1','card2','card3','card4','card5','card6','addr1','addr2','P_emaildomain','R_emaildomain','DeviceType','DeviceInfo']
for i in range(1,10):
    cat_fea.append('M'+str(i))
for i in range(12,39):
    cat_fea.append('id_'+str(i))

num_fea = []
for fea in train.columns:
    if fea not in ['TransactionID','isFraud'] and fea not in cat_fea:
        num_fea.append(fea)


In [ ]:
print(f"Train shape: {train.shape}")
print(f"Test shape: {test.shape}")
print(f"\nFraud rate: {train['isFraud'].mean():.2%}")
print(f"Total fraud cases: {train['isFraud'].sum():,}")
print(f"Total normal cases: {(train['isFraud']==0).sum():,}")

Train shape: (590540, 434)
Test shape: (506691, 433)

Fraud rate: 3.50%
Total fraud cases: 20,663
Total normal cases: 569,877


In [ ]:
test.columns  = test.columns.str.replace('-', '_')

In [ ]:
# define features — everything except ID and label
features = [c for c in train.columns
            if c not in ['TransactionID', 'isFraud']]

# time based split — 80% train, 20% validation
split = train['TransactionDT'].quantile(0.80)
tr = train[train['TransactionDT'] <= split].copy()
val = train[train['TransactionDT'] > split].copy()

X_tr  = tr[features]
y_tr  = tr['isFraud']
X_val = val[features]
y_val = val['isFraud']

print(f"Train size: {len(X_tr):,}")
print(f"Val size:   {len(X_val):,}")
print(f"Fraud rate in train: {y_tr.mean():.2%}")
print(f"Fraud rate in val:   {y_val.mean():.2%}")

Train size: 472,432
Val size:   118,108
Fraud rate in train: 3.51%
Fraud rate in val:   3.44%


In [ ]:

##### examinate the shift
from sklearn.preprocessing import LabelEncoder

le_dict = {}
unknown_stats = []

for col in cat_fea:
    tr_col   = tr[col].astype(str)
    val_col  = val[col].astype(str)
    test_col = test[col].astype(str)

    # calculate unknown rate BEFORE encoding using raw train values
    known_raw = set(tr_col.unique())  # real values only, no 'unknown' yet

    val_unknown_rate  = sum(v not in known_raw for v in val_col)  / len(val_col)  * 100
    test_unknown_rate = sum(v not in known_raw for v in test_col) / len(test_col) * 100

    unknown_stats.append({
        'feature':        col,
        'train_unique':   tr_col.nunique(),
        'val_unknown_%':  round(val_unknown_rate, 2),
        'test_unknown_%': round(test_unknown_rate, 2),
    })

    # NOW encode — add 'unknown' token
    le = LabelEncoder()
    le.fit(list(tr_col) + ['unknown'])
    tr[col]   = le.transform(tr_col)
    known_enc = set(le.classes_)
    val[col]  = le.transform([v if v in known_enc else 'unknown' for v in val_col])
    test[col] = le.transform([v if v in known_enc else 'unknown' for v in test_col])
    le_dict[col] = le

# show results
unknown_df = pd.DataFrame(unknown_stats).sort_values('test_unknown_%', ascending=False)
print(unknown_df.to_string(index=False))
print("\nEncoding done!")

      feature  train_unique  val_unknown_%  test_unknown_%
        id_31           111           6.83           22.06
        id_13            53           3.10           13.74
        id_30            73           0.12            4.39
   DeviceInfo          1640           0.32            3.39
        card1         12730           1.09            2.22
        id_19           511           0.03            1.12
        id_21           445           0.07            0.48
        id_33           213           0.07            0.32
        id_25           318           0.05            0.09
        id_20           381           0.04            0.09
        addr1           329           0.01            0.04
        card2           501           0.00            0.03
        card3           107           0.01            0.02
        card5           114           0.02            0.01
        addr2            71           0.01            0.01
        id_22            24           0.00            0.

In [ ]:
X_tr  = tr[features]
y_tr  = tr['isFraud']
X_val = val[features]
y_val = val['isFraud']

In [ ]:
import lightgbm as lgb

model = lgb.LGBMClassifier(
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=256,
    random_state=42,
    n_jobs=-1
)

model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50), lgb.log_evaluation(100)]
)

[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 1.088271 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 34376
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 431
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784
Training until validation scores don't improve for 50 rounds
[100]	valid_0's binary_logloss: 0.0867019
[200]	valid_0's binary_logloss: 0.084881
Early stopping, best iteration is:
[207]	valid_0's binary_logloss: 0.0848274


LGBMClassifier(learning_rate=0.05, n_estimators=500, n_jobs=-1, num_leaves=256,
               random_state=42)

In [ ]:
from sklearn.metrics import roc_auc_score

val_pred = model.predict_proba(X_val)[:, 1]
auc = roc_auc_score(y_val, val_pred)
print(f"Validation AUC: {auc:.4f}")

Validation AUC: 0.9208


In [ ]:
import pandas as pd

importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print("=== Top 20 most important features ===")
print(importance.head(20))

print("\n=== Bottom 20 least important features ===")
print(importance.tail(20))

=== Top 20 most important features ===
            feature  importance
3             card1        3508
0     TransactionDT        3222
1    TransactionAmt        2737
4             card2        2487
9             addr1        2299
43              D15        1340
11            dist1        1278
13    P_emaildomain        1166
27              C13        1025
7             card5         946
38              D10         938
32               D4         903
30               D2         859
393           id_02         772
29               D1         763
15               C1         719
36               D8         694
411           id_20         646
39              D11         623
422           id_31         606

=== Bottom 20 least important features ===
    feature  importance
243    V191           0
248    V196           0
80      V28           0
293    V241           0
159    V107           0
169    V117           0
171    V119           0
172    V120           0
165    V113           0
357  

In [ ]:
zero_importance = importance[importance['importance'] == 0]

print(f"Total zero importance features: {len(zero_importance)}")
print("\n=== All zero importance features ===")
print(zero_importance['feature'].tolist())

Total zero importance features: 21

=== All zero importance features ===
['id_29', 'V191', 'V196', 'V28', 'V241', 'V107', 'V117', 'V119', 'V120', 'V113', 'V305', 'V88', 'V89', 'V27', 'V41', 'V65', 'V325', 'V327', 'V68', 'id_16', 'id_27']


In [ ]:
++++++++++++ examination ++++++++++++++++++++++++++

In [ ]:
train_ori = train_transaction.merge(train_identity, on='TransactionID', how = 'left')
test_ori = test_transaction.merge(test_identity, on='TransactionID', how = 'left')
tr_ori = train_ori[train_ori['TransactionDT'] <= split].copy()
val_ori = train_ori[train_ori['TransactionDT'] > split].copy()

In [ ]:
tr_ori['id_29'].fillna("missing").value_counts(normalize=True).mul(100).round(2)


id_29
missing     75.07
Found       13.04
NotFound    11.89
Name: proportion, dtype: float64

In [ ]:
val_ori['id_29'].fillna("missing").value_counts(normalize=True).mul(100).round(2)

id_29
missing     80.37
Found       11.27
NotFound     8.35
Name: proportion, dtype: float64

In [ ]:
delete_cols = ['id_29', 'V191', 'V196', 'V28', 'V241', 'V107', 'V117', 'V119', 'V120', 'V113', 'V305', 'V88', 'V89', 'V27', 'V41', 'V65', 'V325', 'V327', 'V68', 'id_16', 'id_27']
val_ori['card1']


NameError: name 'val_ori' is not defined

In [ ]:

low_cardinality  = [c for c in cat_fea if train[c].nunique() < 50]
high_cardinality = [c for c in cat_fea if train[c].nunique() >= 50]

print(f"Low cardinality  (native): {low_cardinality}")
print(f"High cardinality (encode): {high_cardinality}")

In [ ]:
+++++++++++++ handle the categorical feature ++++++++++++++++++

low_cardinality  = [c for c in cat_fea if train[c].nunique() < 50]
high_cardinality = [c for c in cat_fea if train[c].nunique() >= 50]

print(f"Low cardinality  (native): {low_cardinality}")
print(f"High cardinality (encode): {high_cardinality}")

# low cardinality → native LightGBM
for col in low_cardinality:
    train[col] = train[col].astype('category')
    test[col]  = test[col].astype('category')

# high cardinality → frequency encoding
for col in high_cardinality:
    freq = train[col].value_counts() / len(train)
    train[col] = train[col].map(freq).fillna(0)
    test[col]  = test[col].map(freq).fillna(0)